In [20]:
from transformer_lens import HookedTransformer
import torch as t

device = "mps"
reference_gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
    dtype=t.float32,
    device=device
)

# This builds the input in a way the model will understand
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).to(device)

# Those are the models predictions
logits, cache = reference_gpt2.run_with_cache(tokens)
probs = logits.softmax(dim=-1)

# This is converting it to the next token
next_token = logits[0, -1].argmax(dim=-1)
next_token_string = reference_gpt2.to_string(next_token)
print(next_token_string)

Loaded pretrained model gpt2-small into HookedTransformer
.


## Converting it to the next sequence

In [18]:
next_token[None, None]

tensor([[314]])

In [21]:
for i in range(25):
    print(f"{tokens.shape[-1] + 1}th token = {next_token_string!r}")
    # Define new input sequence, by appending the previously generated token
    tokens = t.cat([tokens, next_token[None, None]], dim=-1)
    # Pass our new sequence through the model, to get new output
    logits = reference_gpt2(tokens)
    # Get the predicted token at the end of our sequence
    next_token = logits[0, -1].argmax(dim=-1)
    # Decode and print the result
    next_token_string = reference_gpt2.to_string(next_token)

36th token = '.'
37th token = ' the'
38th token = ' the'
39th token = ' the'
40th token = ' the'
41th token = ' the'
42th token = ' the'
43th token = ' the'
44th token = ' the'
45th token = ' the'
46th token = ' the'
47th token = ' the'
48th token = ' the'
49th token = ' the'
50th token = ' the'
51th token = ' the'
52th token = ' the'
53th token = ' the'
54th token = ' the'
55th token = ' the'
56th token = ' the'
57th token = ' the'
58th token = ' the'
59th token = ' the'
60th token = ' the'


In [56]:
import torch
from transformer_lens import HookedTransformer
torch.set_float32_matmul_precision('high')  # PyTorch 2.1+

# --- configuration ---
model_name = "gpt2-small"
input_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
device_cpu = "cpu"
device_mps = "mps"
model_kwargs = dict(
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
    dtype=torch.float32
)

# --- load model on CPU and MPS ---
model_cpu = HookedTransformer.from_pretrained(model_name, device=device_cpu, **model_kwargs)
model_mps = HookedTransformer.from_pretrained(model_name, device=device_mps, **model_kwargs)

# --- tokenize once ---
tokens = model_cpu.to_tokens(input_text)

# --- forward with cache on CPU ---
_, cache_cpu = model_cpu.run_with_cache(tokens)

# --- forward with cache on MPS ---
_, cache_mps = model_mps.run_with_cache(tokens.to(device_mps))

# --- compare each cache key ---
print("=== Cache differences per key (CPU vs MPS) ===")
for cache_key in cache_cpu.keys():
    # load tensors on CPU to compare
    value_cpu = cache_cpu[cache_key].detach().cpu()
    value_mps = cache_mps[cache_key].detach().cpu()

    diff = value_cpu - value_mps

    mean_abs = diff.abs().mean().item()
    max_abs = diff.abs().max().item()
    l2_norm = torch.norm(diff).item()

    print(f"{cache_key:35s} "
          f"mean={mean_abs:.6e} "
          f"max={max_abs:.6e} "
          f"l2={l2_norm:.6e}")


Loaded pretrained model gpt2-small into HookedTransformer
Loaded pretrained model gpt2-small into HookedTransformer
=== Cache differences per key (CPU vs MPS) ===
hook_embed                          mean=0.000000e+00 max=0.000000e+00 l2=0.000000e+00
hook_pos_embed                      mean=0.000000e+00 max=0.000000e+00 l2=0.000000e+00
blocks.0.hook_resid_pre             mean=0.000000e+00 max=0.000000e+00 l2=0.000000e+00
blocks.0.ln1.hook_scale             mean=5.108970e-09 max=2.980232e-08 l2=5.960464e-08
blocks.0.ln1.hook_normalized        mean=2.428520e-09 max=1.192093e-07 l2=1.041171e-06
blocks.0.attn.hook_q                mean=3.284514e-07 max=4.768372e-06 l2=8.123624e-05
blocks.0.attn.hook_k                mean=2.615901e-07 max=5.245209e-06 l2=6.727275e-05
blocks.0.attn.hook_v                mean=1.580262e-08 max=7.152557e-07 l2=5.153905e-06
blocks.0.attn.hook_attn_scores      mean=nan max=nan l2=nan
blocks.0.attn.hook_pattern          mean=8.850441e-09 max=5.662441e-07 l2=3.60101

On constate que le problème vient du blocks.0.hook_attn_out sur MPS.



In [57]:
W = model_mps.blocks[0].attn.W_O
print("Max element:", W.abs().max())
print("Min nonzero:", W.abs()[W.abs()>0].min())
print("Conditioning approx:", W.abs().max() / W.abs()[W.abs()>0].min())

Max element: tensor(3.3171, device='mps:0', grad_fn=<MaxBackward1>)
Min nonzero: tensor(3.7639e-09, device='mps:0', grad_fn=<MinBackward1>)
Conditioning approx: tensor(8.8131e+08, device='mps:0', grad_fn=<DivBackward0>)


Hypothèse de ChatGPT : Le conditionnement de la matrice serait problématique et donnerait lieu à des erreurs d'arrondis.

In [ ]:
## Cellule inutile et à supprimer?
import torch
from transformer_lens import HookedTransformer
from fancy_einsum import einsum

# --- 1. Configuration et Chargement (Votre code) ---
model_name = "gpt2-small"
input_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
model_kwargs = dict(
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
    dtype=torch.float32
)

print("Chargement des modèles...")
model_cpu = HookedTransformer.from_pretrained(model_name, device="cpu", **model_kwargs)
model_mps = HookedTransformer.from_pretrained(model_name, device="mps", **model_kwargs)

# --- 2. Extraction des données (Run with Cache) ---
print("Exécution du modèle pour récupérer le cache...")
tokens_cpu = model_cpu.to_tokens(input_text)
tokens_mps = model_mps.to_tokens(input_text) # Devrait être identique, mais pour être sûr

_, cache_cpu = model_cpu.run_with_cache(tokens_cpu)
_, cache_mps = model_mps.run_with_cache(tokens_mps)

# On cible la couche 0
layer_idx = 0
hook_z_name = f"blocks.{layer_idx}.attn.hook_z"
hook_out_name = f"blocks.{layer_idx}.hook_attn_out"

# Extraction des entrées 'z' (Ce qui sort des têtes d'attention, juste avant W_O)
# Shape: [batch, pos, n_heads, d_head]
z_cpu_input = cache_cpu[hook_z_name]
z_mps_input = cache_mps[hook_z_name]

# Extraction des sorties réelles pour comparaison
out_cpu_real = cache_cpu[hook_out_name]
out_mps_real = cache_mps[hook_out_name]

# --- 3. Ré-implémentation manuelle de la projection W_O ---

def manual_attn_projection(z_input, model_instance):
    """
    Simule la dernière étape du bloc d'attention : 
    Projection des têtes concaténées vers l'espace résiduel.
    Formula: Out = sum(z * W_O) + b_O
    """
    # Récupération des poids de la couche d'attention
    # W_O shape: [n_heads, d_head, d_model]
    W_O = model_instance.blocks[layer_idx].attn.W_O
    b_O = model_instance.blocks[layer_idx].attn.b_O
    
    # Calcul via Einsum (identique à l'implémentation TransformerLens)
    # z: [batch, pos, head, d_head]
    # W_O: [head, d_head, d_model]
    # -> [batch, pos, d_model]
    
    attn_out = einsum(
        "batch pos head d_head, head d_head d_model -> batch pos d_model", 
        z_input, 
        W_O
    )
    
    return attn_out + b_O

print("\n--- Recalcul Manuel de la Projection de Sortie (W_O) ---")

# Calcul sur CPU
recalc_cpu = manual_attn_projection(z_cpu_input, model_cpu)

# Calcul sur MPS (On utilise les inputs MPS et les poids MPS)
# En PyTorch, vous n'avez pas besoin de dire explicitement "exécute cette fonction sur GPU". Si les tenseurs que vous donnez à la fonction (z_mps_input et model_mps.W_O) sont stockés sur le device mps, alors PyTorch exécute automatiquement l'opération sur MPS.
recalc_mps = manual_attn_projection(z_mps_input, model_mps)


# --- 4. Vérification et Comparaison ---

def print_diff(name, tensor1, tensor2):
    # On ramène tout sur CPU pour le calcul de la diff
    t1 = tensor1.cpu()
    t2 = tensor2.cpu()
    diff = (t1 - t2).abs()
    print(f"Écart {name}:")
    print(f"  Max Diff : {diff.max().item():.6f}")
    print(f"  Mean Diff: {diff.mean().item():.8f}")

# A. Vérification de 'z' (L'entrée)
# Si ça c'est bas, le problème n'est PAS dans le calcul des queries/keys/values/softmax
print_diff("Input 'z' (MPS vs CPU)", z_mps_input, z_cpu_input)

# B. Vérification Sanity Check (Notre code manuel vs TransformerLens interne)
# Cela vérifie que notre fonction 'manual_attn_projection' est correcte
print("-" * 30)
print_diff("Sanity Check CPU (Manuel vs Real)", recalc_cpu, out_cpu_real)
print_diff("Sanity Check MPS (Manuel vs Real)", recalc_mps, out_mps_real)

# C. LE TEST ULTIME : L'écart sur la sortie recalculée
print("-" * 30)
print("LE COUPABLE :")
print_diff("Output 'attn_out' (Manuel MPS vs Manuel CPU)", recalc_mps, recalc_cpu)

# D. Analyse des valeurs brutes pour comprendre l'explosion
print("-" * 30)
print(f"Max value in CPU result: {recalc_cpu.abs().max().item():.4f}")
print(f"Max value in MPS result: {recalc_mps.abs().max().item():.4f}")

Chargement des modèles...
Loaded pretrained model gpt2-small into HookedTransformer
Loaded pretrained model gpt2-small into HookedTransformer
Exécution du modèle pour récupérer le cache...

--- Recalcul Manuel de la Projection de Sortie (W_O) ---
Écart Input 'z' (MPS vs CPU):
  Max Diff : 0.000001
  Mean Diff: 0.00000003
------------------------------
Écart Sanity Check CPU (Manuel vs Real):
  Max Diff : 0.000024
  Mean Diff: 0.00000021
Écart Sanity Check MPS (Manuel vs Real):
  Max Diff : 12.949443
  Mean Diff: 0.65080404
------------------------------
LE COUPABLE :
Écart Output 'attn_out' (Manuel MPS vs Manuel CPU):
  Max Diff : 0.000007
  Mean Diff: 0.00000014
------------------------------
Max value in CPU result: 14.6621
Max value in MPS result: 14.6621


Pourquoi cette différence ?TransformerLens utilise nn.Linear (qui utilise addmm en backend) pour la projection $W_O$.Votre code manuel utilise einsum.Sur le processeur MPS (Apple Silicon), il semble que l'implémentation de addmm (multiplication matricielle standard) soit instable ou buggée pour cette dimension spécifique, alors que einsum (qui décompose le calcul différemment) reste stable.


# Reproduction du problème avec nn.Linear

On a exactement le soucis, le résultat est le même avec nn.Linear sur MPS comme sur CPU. 
Conclusion : Le problème ne vient pas de la libraire, mais de Pytorch lui-même.

## Preuve 

In [3]:
import torch
from transformer_lens import HookedTransformer, utils
from torch import nn
from fancy_einsum import einsum

model_name = "gpt2-small"
model_kwargs = dict(fold_ln=False, center_unembed=False, center_writing_weights=False, dtype=torch.float32)
device = "mps"

print("Chargement des modèles...")
model_cpu = HookedTransformer.from_pretrained(model_name, device="cpu", **model_kwargs)
model_mps = HookedTransformer.from_pretrained(model_name, device="mps", **model_kwargs)

input_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens_cpu = model_cpu.to_tokens(input_text)
tokens_mps = model_mps.to_tokens(input_text)

_, cache_cpu = model_cpu.run_with_cache(tokens_cpu)
_, cache_mps = model_mps.run_with_cache(tokens_mps)
print("Modèles executés...")


layer_idx = 0
hook_z_name = f"blocks.{layer_idx}.attn.hook_z"
hook_out_name = f"blocks.{layer_idx}.hook_attn_out"

# Entrées et Sorties réelles
z_mps_input = cache_mps[hook_z_name]
out_mps_real = cache_mps[hook_out_name]
out_cpu_real = cache_cpu[hook_out_name]

# --- 5. Exécution ---
def run_linear_projection(z_input, model):
    """
    Exécute le calcul  via nn.Linear, en aplatissant l'entrée.
    """
    # Récupération des poids (W_O) et du biais (b_O) de la couche d'attention
    W_O = model.blocks[layer_idx].attn.W_O
    b_O = model.blocks[layer_idx].attn.b_O

    # Dimensions de GPT-2 small
    d_model = 768 
    n_heads = 12
    d_head = 64
    d_in = n_heads * d_head # = 768

    W_O_linear_format = W_O.flatten(start_dim=0, end_dim=1).T

    # 3. Création du module nn.Linear sur MPS
    linear_module = nn.Linear(in_features=d_in, out_features=d_model, bias=True).to(device)

    # 4. Injection des poids
    # nn.Linear.weight est de taille [d_out, d_in] = [768, 768]
    linear_module.weight.data = W_O_linear_format 
    linear_module.bias.data = b_O

    # z_input shape: [batch, pos, n_heads, d_head]
    # On aplatit les dimensions des têtes: [batch, pos, 768]
    z_flat = z_input.flatten(start_dim=2)
    
    # Exécution : [batch, pos, 768] -> nn.Linear -> [batch, pos, 768]
    return linear_module(z_flat)

def manual_attn_projection(z_input, model_instance):
    layer_idx = 0
    W_O = model_instance.blocks[layer_idx].attn.W_O
    b_O = model_instance.blocks[layer_idx].attn.b_O
    
    # Rappel : Puisque z_input et W_O sont sur MPS, ceci s'exécute sur MPS !
    attn_out = einsum(
        "batch pos head d_head, head d_head d_model -> batch pos d_model", 
        z_input, 
        W_O
    )
    return attn_out + b_O

def manual_attn_projection_matmul(z_input, model_instance):
    """
    Reproduit la projection W_O en utilisant flatten + matmul.
    C'est l'opération standard qui semble instable sur MPS.
    """
    # 1. Récupération des poids [n_heads, d_head, d_model]
    W_O = model_instance.blocks[layer_idx].attn.W_O
    b_O = model_instance.blocks[layer_idx].attn.b_O
    
    # 2. APLATISSEMENT (Flattening) des poids
    # On fusionne (heads * d_head) pour obtenir une matrice 2D [d_model_in, d_model_out]
    # Shape devient : [768, 768] (pour GPT-2 small)
    W_O_flat = W_O.flatten(start_dim=0, end_dim=1)
    
    # 3. APLATISSEMENT de l'entrée z
    # z input shape: [batch, pos, n_heads, d_head]
    # On fusionne les deux dernières dimensions
    # Shape devient : [batch, pos, 768]
    z_flat = z_input.flatten(start_dim=2)
    
    # 4. MULTIPLICATION MATRICIELLE (Matmul / Addmm)
    # C'est ici que MPS optimise potentiellement mal
    # [batch, pos, 768] x [768, 768] -> [batch, pos, 768]
    out = torch.matmul(z_flat, W_O_flat)
    
    return out + b_O

print("\n--- Test de reproduction avec nn.Linear sur MPS ---")

# Calcul manuel sur MPS avec nn.Linear et einsum
recalc_mps_linear = run_linear_projection(z_mps_input, model_mps)
recalc_einsum = manual_attn_projection(z_mps_input, model_mps)
recalc_einsum = recalc_einsum.to("cpu")
recalc_matmul = manual_attn_projection_matmul(z_mps_input, model_mps)
recalc_matmul = recalc_matmul.to("cpu")

# --- 6. Vérification de la reproduction du Bug ---

# Si l'écart est proche de 0, on a réussi à reproduire le bug de la librairie.
diff_repro_linear = (recalc_mps_linear - out_mps_real).abs()
diff_repro_einsum = (recalc_einsum - out_cpu_real).abs()
diff_repro_matmul = (recalc_matmul - out_cpu_real).abs()

print(f"Max Diff (Manuel nn.Linear vs Real MPS Output): {diff_repro_linear.max().item():.6f}")
print(f"Max Diff (Manuel einsum vs MPS Output): {diff_repro_einsum.max().item():.6f}")

if diff_repro_linear.max().item() < 0.1:
    print("\n✅ SUCCÈS DE LA REPRODUCTION DU BUG !")
    print("Cela confirme que la couche 'nn.Linear' standard, telle qu'implémentée dans PyTorch sur MPS, est la source de l'instabilité.")
    print("Elle ne gère pas de manière stable la sommation de cette grande multiplication matricielle.")
else:
    print("\n❌ ÉCHEC. L'implémentation interne de TransformerLens est encore plus complexe.")

if diff_repro_einsum.max().item() < 0.1:
    print("\n✅ VICTOIRE ! L'implémentation 'Einsum' sur MPS corrige le problème.")
    print("Le bug vient de l'implémentation standard de nn.Linear/addmm sur Mac.")
else:
    print("\n❌ Toujours une erreur. Le mystère s'épaissit.")

if diff_repro_matmul.max().item() < 0.1:
    print("\n✅ VICTOIRE ! L'implémentation 'Matmul' sur MPS corrige le problème.")
    print("Le bug vient de l'implémentation standard de nn.Linear/addmm sur Mac.")
else:
    print("\n❌ Toujours une erreur. Le mystère s'épaissit.")

Chargement des modèles...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Loaded pretrained model gpt2-small into HookedTransformer
Modèles executés...

--- Test de reproduction avec nn.Linear sur MPS ---
Max Diff (Manuel nn.Linear vs Real MPS Output): 0.000000
Max Diff (Manuel einsum vs MPS Output): 0.000023

✅ SUCCÈS DE LA REPRODUCTION DU BUG !
Cela confirme que la couche 'nn.Linear' standard, telle qu'implémentée dans PyTorch sur MPS, est la source de l'instabilité.
Elle ne gère pas de manière stable la sommation de cette grande multiplication matricielle.

✅ VICTOIRE ! L'implémentation 'Einsum' sur MPS corrige le problème.
Le bug vient de l'implémentation standard de nn.Linear/addmm sur Mac.

✅ VICTOIRE ! L'implémentation 'Matmul' sur MPS corrige le problème.
Le bug vient de l'implémentation standard de nn.Linear/addmm sur Mac.


# Creusons nn.Linear

L'objectif est désormais de reproduire le calcul exactement tel que le fait nn.Linear, afin de pouvoir comprendre d'où vient l'erreur


In [13]:
# Exécution du script de débogage pour reproduire exactement nn.Linear
# Ce script sort du notebook pour permettre un débogage plus détaillé

import subprocess
import sys

# Exécuter le script de débogage
result = subprocess.run(
    [sys.executable, "debug_linear.py"],
    cwd=".",
    capture_output=True,
    text=True
)

# Afficher la sortie
print(result.stdout)
if result.stderr:
    print("ERREURS:", file=sys.stderr)
    print(result.stderr, file=sys.stderr)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DÉBOGAGE DE nn.Linear - Reproduction exacte du calcul

1. Chargement des modèles...
Loaded pretrained model gpt2-small into HookedTransformer
Loaded pretrained model gpt2-small into HookedTransformer
2. Tokenisation...
3. Exécution du modèle pour récupérer le cache...

4. Dimensions:
   z shape: torch.Size([1, 35, 12, 64])
   W_O shape: torch.Size([12, 64, 768])
   b_O shape: torch.Size([768])
   out shape: torch.Size([1, 35, 768])

MÉTHODE 0: Utilisation directe de nn.Linear (référence)

Différence (nn.Linear CPU vs réel CPU):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

Différence (nn.Linear MPS vs réel MPS):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

Différence (nn.Linear CPU vs nn.Linear MPS):
   Max: 1.2949441910e+01
   Mean: 6.5080404282e-01

MÉTHODE 1: Utilisation de torch.nn.functional.linear

Différence (F.linear CPU vs réel CPU):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

Différence (F.linear MPS vs réel MPS):
   Max: 0.0000000000e+00
   Mean: 0.00000

ERREURS:
/Users/geoffreyvoyer/dev/ARENA_3.0/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/geoffreyvoyer/dev/ARENA_3.0/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This 

In [14]:
from debug_linear import debug_linear_computation

# Utiliser les modèles et caches déjà chargés
results = debug_linear_computation(
    model_cpu=model_cpu,
    model_mps=model_mps,
    cache_cpu=cache_cpu,
    cache_mps=cache_mps,
    layer_idx=0,
    verbose=True
)


Résultats du débogage:
  nn_linear       - Max diff: 1.2949441910e+01, Mean diff: 6.5080404282e-01
  F_linear        - Max diff: 1.2949441910e+01, Mean diff: 6.5080404282e-01
  addmm           - Max diff: 2.0027160645e-05, Mean diff: 2.6708488576e-07
  matmul          - Max diff: 1.5497207642e-06, Mean diff: 1.7729824719e-07


Les résultats ci-dessus indiquent : 
- Que addmm fonctionne exactement pareil sur CPU que sur MPS (ce qui est une bonne chose)
- Que addmm fournit exactement le même résultat sur que nn.Linear sur MPS, ce qui suggère que c'est bien ce qui est utilisé dans le cas du device MPS
- Que le matmul MPS cursor fournit exactement le même résultat sur MPS, ce qui laisse penser que l'implémentation est cohérente.
- Le matmul manuel avec un transpose fournit en revanche approximativement le même résultat que le CPU. 


In [15]:
# Fonction corrigée de reproduce_with_matmul
# Le problème était la transposition de W_O_flat.T - ici on ne transpose pas

def reproduce_with_matmul_corrected(z_input, W_O, b_O, device):
    """
    Version corrigée de reproduce_with_matmul.
    Ne transpose PAS W_O_flat, ce qui correspond à l'opération einsum correcte.
    """
    # Aplatir z: [batch, pos, n_heads, d_head] -> [batch*pos, n_heads*d_head]
    batch, pos, n_heads, d_head = z_input.shape
    z_flat = z_input.flatten(start_dim=0, end_dim=1)  # [batch*pos, n_heads, d_head]
    z_flat = z_flat.flatten(start_dim=1)  # [batch*pos, n_heads*d_head]
    
    # Aplatir W_O: [n_heads, d_head, d_model] -> [n_heads*d_head, d_model]
    W_O_flat = W_O.flatten(start_dim=0, end_dim=1)  # [n_heads*d_head, d_model]
    
    # Matmul SANS transposition: [batch*pos, n_heads*d_head] @ [n_heads*d_head, d_model] -> [batch*pos, d_model]
    output_flat = torch.matmul(z_flat, W_O_flat)  # PAS de .T ici!
    output_flat = output_flat + b_O  # Broadcasting
    
    # Reshape: [batch*pos, d_model] -> [batch, pos, d_model]
    output = output_flat.reshape(batch, pos, -1)
    return output


In [16]:
# Comparaison détaillée entre les 3 fonctions matmul
# 1. reproduce_with_matmul (avec transposition - INCORRECT)
# 2. reproduce_with_matmul_corrected (sans transposition - CORRECT)
# 3. manual_attn_projection_matmul (sans transposition, structure différente - CORRECT)

from debug_linear import reproduce_with_matmul
import torch

# Extraction des données (en utilisant les variables déjà définies)
layer_idx = 0
hook_z_name = f"blocks.{layer_idx}.attn.hook_z"
hook_out_name = f"blocks.{layer_idx}.hook_attn_out"

z_cpu = cache_cpu[hook_z_name]
z_mps = cache_mps[hook_z_name]
out_cpu_real = cache_cpu[hook_out_name]
out_mps_real = cache_mps[hook_out_name]

W_O_cpu = model_cpu.blocks[layer_idx].attn.W_O
b_O_cpu = model_cpu.blocks[layer_idx].attn.b_O
W_O_mps = model_mps.blocks[layer_idx].attn.W_O
b_O_mps = model_mps.blocks[layer_idx].attn.b_O

print("=" * 80)
print("COMPARAISON DES 3 FONCTIONS MATMUL")
print("1. reproduce_with_matmul (avec .T - INCORRECT)")
print("2. reproduce_with_matmul_corrected (sans .T - CORRECT)")
print("3. manual_attn_projection_matmul (sans .T, structure différente - CORRECT)")
print("=" * 80)

# ============================================================================
# TESTS SUR CPU
# ============================================================================

print("\n" + "=" * 80)
print("TESTS SUR CPU")
print("=" * 80)

# reproduce_with_matmul sur CPU
out_cpu_reproduce = reproduce_with_matmul(z_cpu, W_O_cpu, b_O_cpu, "cpu")
diff_cpu_reproduce_vs_real = (out_cpu_reproduce - out_cpu_real).abs()
print(f"\nreproduce_with_matmul (CPU vs réel CPU):")
print(f"   Max: {diff_cpu_reproduce_vs_real.max().item():.10e}")
print(f"   Mean: {diff_cpu_reproduce_vs_real.mean().item():.10e}")

# reproduce_with_matmul_corrected sur CPU
out_cpu_corrected = reproduce_with_matmul_corrected(z_cpu, W_O_cpu, b_O_cpu, "cpu")
diff_cpu_corrected_vs_real = (out_cpu_corrected - out_cpu_real).abs()
print(f"\nreproduce_with_matmul_corrected (CPU vs réel CPU):")
print(f"   Max: {diff_cpu_corrected_vs_real.max().item():.10e}")
print(f"   Mean: {diff_cpu_corrected_vs_real.mean().item():.10e}")

# manual_attn_projection_matmul sur CPU
out_cpu_manual = manual_attn_projection_matmul(z_cpu, model_cpu)
diff_cpu_manual_vs_real = (out_cpu_manual - out_cpu_real).abs()
print(f"\nmanual_attn_projection_matmul (CPU vs réel CPU):")
print(f"   Max: {diff_cpu_manual_vs_real.max().item():.10e}")
print(f"   Mean: {diff_cpu_manual_vs_real.mean().item():.10e}")

# Comparaisons entre les fonctions sur CPU
diff_cpu_reproduce_vs_corrected = (out_cpu_reproduce - out_cpu_corrected).abs()
print(f"\nDifférence reproduce_with_matmul vs corrected (CPU):")
print(f"   Max: {diff_cpu_reproduce_vs_corrected.max().item():.10e}")
print(f"   Mean: {diff_cpu_reproduce_vs_corrected.mean().item():.10e}")

diff_cpu_corrected_vs_manual = (out_cpu_corrected - out_cpu_manual).abs()
print(f"\nDifférence corrected vs manual (CPU):")
print(f"   Max: {diff_cpu_corrected_vs_manual.max().item():.10e}")
print(f"   Mean: {diff_cpu_corrected_vs_manual.mean().item():.10e}")

diff_cpu_reproduce_vs_manual = (out_cpu_reproduce - out_cpu_manual).abs()
print(f"\nDifférence reproduce_with_matmul vs manual (CPU):")
print(f"   Max: {diff_cpu_reproduce_vs_manual.max().item():.10e}")
print(f"   Mean: {diff_cpu_reproduce_vs_manual.mean().item():.10e}")

# ============================================================================
# TESTS SUR MPS
# ============================================================================

print("\n" + "=" * 80)
print("TESTS SUR MPS")
print("=" * 80)

# reproduce_with_matmul sur MPS
out_mps_reproduce = reproduce_with_matmul(z_mps, W_O_mps, b_O_mps, "mps")
diff_mps_reproduce_vs_real = (out_mps_reproduce - out_mps_real).abs()
print(f"\nreproduce_with_matmul (MPS vs réel MPS):")
print(f"   Max: {diff_mps_reproduce_vs_real.max().item():.10e}")
print(f"   Mean: {diff_mps_reproduce_vs_real.mean().item():.10e}")

# reproduce_with_matmul_corrected sur MPS
out_mps_corrected = reproduce_with_matmul_corrected(z_mps, W_O_mps, b_O_mps, "mps")
diff_mps_corrected_vs_real = (out_mps_corrected - out_mps_real).abs()
print(f"\nreproduce_with_matmul_corrected (MPS vs réel MPS):")
print(f"   Max: {diff_mps_corrected_vs_real.max().item():.10e}")
print(f"   Mean: {diff_mps_corrected_vs_real.mean().item():.10e}")

# manual_attn_projection_matmul sur MPS
out_mps_manual = manual_attn_projection_matmul(z_mps, model_mps)
diff_mps_manual_vs_real = (out_mps_manual - out_mps_real).abs()
print(f"\nmanual_attn_projection_matmul (MPS vs réel MPS):")
print(f"   Max: {diff_mps_manual_vs_real.max().item():.10e}")
print(f"   Mean: {diff_mps_manual_vs_real.mean().item():.10e}")

# Comparaisons entre les fonctions sur MPS
diff_mps_reproduce_vs_corrected = (out_mps_reproduce - out_mps_corrected).abs()
print(f"\nDifférence reproduce_with_matmul vs corrected (MPS):")
print(f"   Max: {diff_mps_reproduce_vs_corrected.max().item():.10e}")
print(f"   Mean: {diff_mps_reproduce_vs_corrected.mean().item():.10e}")

diff_mps_corrected_vs_manual = (out_mps_corrected - out_mps_manual).abs()
print(f"\nDifférence corrected vs manual (MPS):")
print(f"   Max: {diff_mps_corrected_vs_manual.max().item():.10e}")
print(f"   Mean: {diff_mps_corrected_vs_manual.mean().item():.10e}")

diff_mps_reproduce_vs_manual = (out_mps_reproduce - out_mps_manual).abs()
print(f"\nDifférence reproduce_with_matmul vs manual (MPS):")
print(f"   Max: {diff_mps_reproduce_vs_manual.max().item():.10e}")
print(f"   Mean: {diff_mps_reproduce_vs_manual.mean().item():.10e}")

# ============================================================================
# COMPARAISON CROSS-DEVICE
# ============================================================================

print("\n" + "=" * 80)
print("COMPARAISON CROSS-DEVICE")
print("=" * 80)

# reproduce_with_matmul: CPU vs MPS
diff_reproduce_cpu_mps = (out_cpu_reproduce - out_mps_reproduce.cpu()).abs()
print(f"\nreproduce_with_matmul (CPU vs MPS):")
print(f"   Max: {diff_reproduce_cpu_mps.max().item():.10e}")
print(f"   Mean: {diff_reproduce_cpu_mps.mean().item():.10e}")

# reproduce_with_matmul_corrected: CPU vs MPS
diff_corrected_cpu_mps = (out_cpu_corrected - out_mps_corrected.cpu()).abs()
print(f"\nreproduce_with_matmul_corrected (CPU vs MPS):")
print(f"   Max: {diff_corrected_cpu_mps.max().item():.10e}")
print(f"   Mean: {diff_corrected_cpu_mps.mean().item():.10e}")

# manual_attn_projection_matmul: CPU vs MPS
diff_manual_cpu_mps = (out_cpu_manual - out_mps_manual.cpu()).abs()
print(f"\nmanual_attn_projection_matmul (CPU vs MPS):")
print(f"   Max: {diff_manual_cpu_mps.max().item():.10e}")
print(f"   Mean: {diff_manual_cpu_mps.mean().item():.10e}")

# ============================================================================
# ANALYSE DÉTAILLÉE DES DIFFÉRENCES
# ============================================================================

print("\n" + "=" * 80)
print("ANALYSE DÉTAILLÉE")
print("=" * 80)

# Vérifier les dimensions intermédiaires
batch, pos, n_heads, d_head = z_cpu.shape
d_model = W_O_cpu.shape[-1]

z_flat_reproduce = z_cpu.flatten(start_dim=0, end_dim=1).flatten(start_dim=1)
z_flat_manual = z_cpu.flatten(start_dim=2)
W_O_flat = W_O_cpu.flatten(start_dim=0, end_dim=1)

print(f"\n1. Dimensions intermédiaires:")
print(f"   z_flat (reproduce): {z_flat_reproduce.shape}")
print(f"   z_flat (manual): {z_flat_manual.shape}")
print(f"   W_O_flat: {W_O_flat.shape}")
print(f"   W_O_flat.T: {W_O_flat.T.shape}")

# Vérifier si les calculs sont équivalents
matmul_reproduce = torch.matmul(z_flat_reproduce, W_O_flat.T)  # [batch*pos, d_model] - avec .T
matmul_corrected = torch.matmul(z_flat_reproduce, W_O_flat)  # [batch*pos, d_model] - sans .T
matmul_manual = torch.matmul(z_flat_manual, W_O_flat)  # [batch, pos, d_model]

# Reshape pour comparer
matmul_reproduce_reshaped = matmul_reproduce.reshape(batch, pos, -1)
matmul_corrected_reshaped = matmul_corrected.reshape(batch, pos, -1)

diff_matmul_reproduce_vs_manual = (matmul_reproduce_reshaped - matmul_manual).abs()
diff_matmul_corrected_vs_manual = (matmul_corrected_reshaped - matmul_manual).abs()

print(f"\n2. Différence dans le matmul seul (avant ajout du bias):")
print(f"   reproduce (avec .T) vs manual:")
print(f"      Max: {diff_matmul_reproduce_vs_manual.max().item():.10e}")
print(f"      Mean: {diff_matmul_reproduce_vs_manual.mean().item():.10e}")
print(f"   corrected (sans .T) vs manual:")
print(f"      Max: {diff_matmul_corrected_vs_manual.max().item():.10e}")
print(f"      Mean: {diff_matmul_corrected_vs_manual.mean().item():.10e}")

# Vérifier les valeurs statistiques
print(f"\n3. Statistiques des résultats:")
print(f"   reproduce_with_matmul (CPU):")
print(f"      Min: {out_cpu_reproduce.min().item():.6f}")
print(f"      Max: {out_cpu_reproduce.max().item():.6f}")
print(f"      Mean: {out_cpu_reproduce.mean().item():.6f}")
print(f"      Std: {out_cpu_reproduce.std().item():.6f}")

print(f"\n   reproduce_with_matmul_corrected (CPU):")
print(f"      Min: {out_cpu_corrected.min().item():.6f}")
print(f"      Max: {out_cpu_corrected.max().item():.6f}")
print(f"      Mean: {out_cpu_corrected.mean().item():.6f}")
print(f"      Std: {out_cpu_corrected.std().item():.6f}")

print(f"\n   manual_attn_projection_matmul (CPU):")
print(f"      Min: {out_cpu_manual.min().item():.6f}")
print(f"      Max: {out_cpu_manual.max().item():.6f}")
print(f"      Mean: {out_cpu_manual.mean().item():.6f}")
print(f"      Std: {out_cpu_manual.std().item():.6f}")

print(f"\n   reproduce_with_matmul (MPS):")
print(f"      Min: {out_mps_reproduce.min().item():.6f}")
print(f"      Max: {out_mps_reproduce.max().item():.6f}")
print(f"      Mean: {out_mps_reproduce.mean().item():.6f}")
print(f"      Std: {out_mps_reproduce.std().item():.6f}")

print(f"\n   reproduce_with_matmul_corrected (MPS):")
print(f"      Min: {out_mps_corrected.min().item():.6f}")
print(f"      Max: {out_mps_corrected.max().item():.6f}")
print(f"      Mean: {out_mps_corrected.mean().item():.6f}")
print(f"      Std: {out_mps_corrected.std().item():.6f}")

print(f"\n   manual_attn_projection_matmul (MPS):")
print(f"      Min: {out_mps_manual.min().item():.6f}")
print(f"      Max: {out_mps_manual.max().item():.6f}")
print(f"      Mean: {out_mps_manual.mean().item():.6f}")
print(f"      Std: {out_mps_manual.std().item():.6f}")

print("\n" + "=" * 80)
print("FIN DE LA COMPARAISON")
print("=" * 80)


COMPARAISON DES 3 FONCTIONS MATMUL
1. reproduce_with_matmul (avec .T - INCORRECT)
2. reproduce_with_matmul_corrected (sans .T - CORRECT)
3. manual_attn_projection_matmul (sans .T, structure différente - CORRECT)

TESTS SUR CPU

reproduce_with_matmul (CPU vs réel CPU):
   Max: 1.2949441910e+01
   Mean: 6.5080404282e-01

reproduce_with_matmul_corrected (CPU vs réel CPU):
   Max: 2.2888183594e-05
   Mean: 1.6383867774e-07

manual_attn_projection_matmul (CPU vs réel CPU):
   Max: 2.2888183594e-05
   Mean: 1.6383867774e-07

Différence reproduce_with_matmul vs corrected (CPU):
   Max: 1.2949441910e+01
   Mean: 6.5080398321e-01

Différence corrected vs manual (CPU):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

Différence reproduce_with_matmul vs manual (CPU):
   Max: 1.2949441910e+01
   Mean: 6.5080398321e-01

TESTS SUR MPS

reproduce_with_matmul (MPS vs réel MPS):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

reproduce_with_matmul_corrected (MPS vs réel MPS):
   Max: 1.294944572

## Recherche de l'origine du transpose

- Je veux vérifier si le transpose vient d'une différence entre w_0_cpu vs w_0_mps
- Si j'ai bien un écart important entre le CPU et le MPS avec exactement les mêmes entrées (z et w0 du cpu par exemple)

In [19]:
# Vérification: La différence vient-elle des poids W_O ou de l'implémentation?
# On compare W_O_cpu vs W_O_mps et on teste avec les mêmes entrées

import torch
import torch.nn as nn
from debug_linear import reproduce_with_matmul, reproduce_with_nn_linear

# Extraction des données (en utilisant les variables déjà définies)
layer_idx = 0
hook_z_name = f"blocks.{layer_idx}.attn.hook_z"

z_cpu = cache_cpu[hook_z_name]
z_mps = cache_mps[hook_z_name]

W_O_cpu = model_cpu.blocks[layer_idx].attn.W_O
b_O_cpu = model_cpu.blocks[layer_idx].attn.b_O
W_O_mps = model_mps.blocks[layer_idx].attn.W_O
b_O_mps = model_mps.blocks[layer_idx].attn.b_O

print("=" * 80)
print("COMPARAISON DES POIDS W_O ET DES ENTRÉES Z")
print("=" * 80)

# ============================================================================
# 1. Comparaison des poids W_O entre CPU et MPS
# ============================================================================

print("\n1. Différence dans les poids W_O (CPU vs MPS):")
W_O_diff = (W_O_cpu - W_O_mps.cpu()).abs()
print(f"   Max: {W_O_diff.max().item():.10e}")
print(f"   Mean: {W_O_diff.mean().item():.10e}")
print(f"   Min: {W_O_diff.min().item():.10e}")
print(f"   Nombre d'éléments différents (> 1e-10): {(W_O_diff > 1e-10).sum().item()}")
print(f"   Nombre d'éléments différents (> 1e-6): {(W_O_diff > 1e-6).sum().item()}")

# ============================================================================
# 2. Comparaison des biais b_O entre CPU et MPS
# ============================================================================

print("\n2. Différence dans les biais b_O (CPU vs MPS):")
b_O_diff = (b_O_cpu - b_O_mps.cpu()).abs()
print(f"   Max: {b_O_diff.max().item():.10e}")
print(f"   Mean: {b_O_diff.mean().item():.10e}")
print(f"   Min: {b_O_diff.min().item():.10e}")

# ============================================================================
# 3. Comparaison des entrées z entre CPU et MPS
# ============================================================================

print("\n3. Différence dans les entrées z (CPU vs MPS):")
z_diff = (z_cpu - z_mps.cpu()).abs()
print(f"   Max: {z_diff.max().item():.10e}")
print(f"   Mean: {z_diff.mean().item():.10e}")
print(f"   Min: {z_diff.min().item():.10e}")

# ============================================================================
# 4. TEST CRUCIAL: Utiliser les MÊMES entrées (CPU) sur CPU et MPS
# ============================================================================

print("\n" + "=" * 80)
print("TEST: Utiliser les MÊMES entrées CPU sur CPU et MPS")
print("=" * 80)

# Transférer z_cpu et W_O_cpu sur MPS
z_cpu_on_mps = z_cpu.to("mps")
W_O_cpu_on_mps = W_O_cpu.to("mps")
b_O_cpu_on_mps = b_O_cpu.to("mps")

# Calculer avec reproduce_with_matmul (avec .T) sur CPU
out_cpu_same_inputs = reproduce_with_matmul(z_cpu, W_O_cpu, b_O_cpu, "cpu")

# Calculer avec reproduce_with_matmul (avec .T) sur MPS avec les MÊMES entrées CPU
out_mps_same_inputs = reproduce_with_matmul(z_cpu_on_mps, W_O_cpu_on_mps, b_O_cpu_on_mps, "mps")

# Comparer les résultats
diff_same_inputs = (out_cpu_same_inputs - out_mps_same_inputs.cpu()).abs()
print(f"\nDifférence (mêmes entrées CPU, calcul sur CPU vs MPS avec reproduce_with_matmul):")
print(f"   Max: {diff_same_inputs.max().item():.10e}")
print(f"   Mean: {diff_same_inputs.mean().item():.10e}")

# Test avec la version corrigée (sans .T)
out_cpu_corrected_same = reproduce_with_matmul_corrected(z_cpu, W_O_cpu, b_O_cpu, "cpu")
out_mps_corrected_same = reproduce_with_matmul_corrected(z_cpu_on_mps, W_O_cpu_on_mps, b_O_cpu_on_mps, "mps")

diff_corrected_same_inputs = (out_cpu_corrected_same - out_mps_corrected_same.cpu()).abs()
print(f"\nDifférence (mêmes entrées CPU, calcul sur CPU vs MPS avec reproduce_with_matmul_corrected):")
print(f"   Max: {diff_corrected_same_inputs.max().item():.10e}")
print(f"   Mean: {diff_corrected_same_inputs.mean().item():.10e}")

# Test avec nn.Linear (la méthode utilisée par PyTorch)
out_cpu_nn_linear_same = reproduce_with_nn_linear(z_cpu, W_O_cpu, b_O_cpu, "cpu")
out_mps_nn_linear_same = reproduce_with_nn_linear(z_cpu_on_mps, W_O_cpu_on_mps, b_O_cpu_on_mps, "mps")

diff_nn_linear_same_inputs = (out_cpu_nn_linear_same - out_mps_nn_linear_same.cpu()).abs()
print(f"\nDifférence (mêmes entrées CPU, calcul sur CPU vs MPS avec nn.Linear):")
print(f"   Max: {diff_nn_linear_same_inputs.max().item():.10e}")
print(f"   Mean: {diff_nn_linear_same_inputs.mean().item():.10e}")

# ============================================================================
# 5. TEST: Utiliser les entrées MPS sur CPU
# ============================================================================

print("\n" + "=" * 80)
print("TEST: Utiliser les entrées MPS sur CPU")
print("=" * 80)

# Transférer z_mps et W_O_mps sur CPU
z_mps_on_cpu = z_mps.cpu()
W_O_mps_on_cpu = W_O_mps.cpu()
b_O_mps_on_cpu = b_O_mps.cpu()

# Calculer sur CPU avec les entrées MPS
out_cpu_mps_inputs = reproduce_with_matmul(z_mps_on_cpu, W_O_mps_on_cpu, b_O_mps_on_cpu, "cpu")
out_mps_mps_inputs = reproduce_with_matmul(z_mps, W_O_mps, b_O_mps, "mps")

diff_mps_inputs = (out_cpu_mps_inputs - out_mps_mps_inputs.cpu()).abs()
print(f"\nDifférence (entrées MPS, calcul sur CPU vs MPS avec reproduce_with_matmul):")
print(f"   Max: {diff_mps_inputs.max().item():.10e}")
print(f"   Mean: {diff_mps_inputs.mean().item():.10e}")

# ============================================================================
# 6. ANALYSE: Comparaison des résultats réels du modèle
# ============================================================================

print("\n" + "=" * 80)
print("ANALYSE: Comparaison avec les résultats réels du modèle")
print("=" * 80)

hook_out_name = f"blocks.{layer_idx}.hook_attn_out"
out_cpu_real = cache_cpu[hook_out_name]
out_mps_real = cache_mps[hook_out_name]

# Différence entre les résultats réels CPU et MPS
diff_real = (out_cpu_real - out_mps_real.cpu()).abs()
print(f"\nDifférence entre les résultats réels (CPU vs MPS):")
print(f"   Max: {diff_real.max().item():.10e}")
print(f"   Mean: {diff_real.mean().item():.10e}")

# Vérifier quelle fonction correspond au résultat réel
diff_cpu_reproduce_vs_real = (out_cpu_same_inputs - out_cpu_real).abs()
diff_mps_reproduce_vs_real = (out_mps_same_inputs.cpu() - out_mps_real.cpu()).abs()

print(f"\nreproduce_with_matmul (avec .T) vs résultat réel:")
print(f"   CPU: Max={diff_cpu_reproduce_vs_real.max().item():.10e}, Mean={diff_cpu_reproduce_vs_real.mean().item():.10e}")
print(f"   MPS: Max={diff_mps_reproduce_vs_real.max().item():.10e}, Mean={diff_mps_reproduce_vs_real.mean().item():.10e}")

diff_cpu_corrected_vs_real = (out_cpu_corrected_same - out_cpu_real).abs()
diff_mps_corrected_vs_real = (out_mps_corrected_same.cpu() - out_mps_real.cpu()).abs()

print(f"\nreproduce_with_matmul_corrected (sans .T) vs résultat réel:")
print(f"   CPU: Max={diff_cpu_corrected_vs_real.max().item():.10e}, Mean={diff_cpu_corrected_vs_real.mean().item():.10e}")
print(f"   MPS: Max={diff_mps_corrected_vs_real.max().item():.10e}, Mean={diff_mps_corrected_vs_real.mean().item():.10e}")

diff_cpu_nn_linear_vs_real = (out_cpu_nn_linear_same - out_cpu_real).abs()
diff_mps_nn_linear_vs_real = (out_mps_nn_linear_same.cpu() - out_mps_real.cpu()).abs()

print(f"\nnn.Linear vs résultat réel:")
print(f"   CPU: Max={diff_cpu_nn_linear_vs_real.max().item():.10e}, Mean={diff_cpu_nn_linear_vs_real.mean().item():.10e}")
print(f"   MPS: Max={diff_mps_nn_linear_vs_real.max().item():.10e}, Mean={diff_mps_nn_linear_vs_real.mean().item():.10e}")

print("\n" + "=" * 80)
print("CONCLUSION:")
print("=" * 80)
print("Si les poids W_O sont identiques (diff ≈ 0) mais que les résultats diffèrent,")
print("alors la différence vient de l'implémentation (transposition) et non des poids.")
print("=" * 80)


COMPARAISON DES POIDS W_O ET DES ENTRÉES Z

1. Différence dans les poids W_O (CPU vs MPS):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00
   Min: 0.0000000000e+00
   Nombre d'éléments différents (> 1e-10): 0
   Nombre d'éléments différents (> 1e-6): 0

2. Différence dans les biais b_O (CPU vs MPS):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00
   Min: 0.0000000000e+00

3. Différence dans les entrées z (CPU vs MPS):
   Max: 5.6624412537e-07
   Mean: 2.5750699351e-08
   Min: 0.0000000000e+00

TEST: Utiliser les MÊMES entrées CPU sur CPU et MPS

Différence (mêmes entrées CPU, calcul sur CPU vs MPS avec reproduce_with_matmul):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

Différence (mêmes entrées CPU, calcul sur CPU vs MPS avec reproduce_with_matmul_corrected):
   Max: 0.0000000000e+00
   Mean: 0.0000000000e+00

Différence (mêmes entrées CPU, calcul sur CPU vs MPS avec nn.Linear):
   Max: 1.2949441910e+01
   Mean: 6.5080404282e-01

TEST: Utiliser les entrées MPS sur CPU

Di